Combining our datasets

In [9]:
# 1808 dataset

from roboflow import Roboflow
rf = Roboflow(api_key="5XEfDZRIIjMqbazGL6nS")
project = rf.workspace("bachelor-d6dut").project("car-detection-bachelor")
version = project.version(3)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Car-detection-bachelor-3 in yolov8:: 100%|██████████| 3628/3628 [00:02<00:00, 1476.00it/s]


In [10]:
# 998 dataset

rf = Roboflow(api_key="zolhGuu1snsuy5zeAJsS")
project = rf.workspace("bachelor-jvyhh").project("superstar-car-project")
version = project.version(2)
dataset = version.download("yolov8")
                
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to superstar-car-project-2 in yolov8:: 100%|██████████| 2002/2002 [00:01<00:00, 1248.85it/s]


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="VcIz6aIo01dcvjkU8AMO")
project = rf.workspace("labeling-2").project("labeling-2-hoswc")
version = project.version(1)
dataset = version.download("yolov8")
                

In [14]:
# merging the datasets
import os
import shutil
import random

# Paths to your datasets
datasets_folder = os.path.join("datasets")

# Output merged dataset
merged = "merged_dataset"

# Create merged folders if they don't exist
os.makedirs(os.path.join(merged, "images"), exist_ok=True)
os.makedirs(os.path.join(merged, "labels"), exist_ok=True)

dataset_paths = [
    os.path.join(datasets_folder, name)
    for name in os.listdir(datasets_folder)
    if os.path.isdir(os.path.join(datasets_folder, name))
]

# Function to copy files from each dataset
def copy_dataset(source_folder, merged_folder):
    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(source_folder, split, 'images')
        lbl_dir = os.path.join(source_folder, split, 'labels')
        if os.path.exists(img_dir):
            for file in os.listdir(img_dir):
                shutil.copy(os.path.join(img_dir, file), os.path.join(merged_folder, 'images', file))
        if os.path.exists(lbl_dir):
            for file in os.listdir(lbl_dir):
                shutil.copy(os.path.join(lbl_dir, file), os.path.join(merged_folder, 'labels', file))

# Copy all datasets
for dataset in dataset_paths:
    copy_dataset(dataset, merged)

In [15]:
# Split into train/val/test
image_files = os.listdir(os.path.join(merged, "images"))
random.shuffle(image_files)

total = len(image_files)
train_split = int(0.7 * total)
val_split = int(0.85 * total)  # 70% + 15% = 85%

train_files = image_files[:train_split]
val_files = image_files[train_split:val_split]
test_files = image_files[val_split:]

# Create split folders
for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(merged, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(merged, split, 'labels'), exist_ok=True)

# Function to move files into split folders
def move_files(file_list, split):
    for file in file_list:
        img_src = os.path.join(merged, "images", file)
        lbl_src = os.path.join(merged, "labels", file.rsplit('.', 1)[0] + '.txt')  # handles .jpg or .png
        
        img_dst = os.path.join(merged, split, "images", file)
        lbl_dst = os.path.join(merged, split, "labels", file.rsplit('.', 1)[0] + '.txt')

        if os.path.exists(img_src):
            shutil.move(img_src, img_dst)
        if os.path.exists(lbl_src):
            shutil.move(lbl_src, lbl_dst)

# Move files
move_files(train_files, "train")
move_files(val_files, "valid")
move_files(test_files, "test")

# (Optional) Clean up empty original images/labels folders
shutil.rmtree(os.path.join(merged, "images"))
shutil.rmtree(os.path.join(merged, "labels"))

print("✅ Dataset merged and split: 70% train, 15% valid, 15% test.")


✅ Dataset merged and split: 70% train, 15% valid, 15% test.
